> **OPTIONAL: not used for the reported results.** The project's detection results come from a
> pretrained DeepFashion2 YOLOv8s-seg model (`src/download_weights.py`), evaluated on our held-out split.
> This notebook is the "train your own detector" path, kept for anyone who wants to fine-tune YOLO11n
> on the same 15k-image subset and compare. It was smoke-tested end to end (1 epoch on 2% of the data) but **never run to completion**.
> Note: its evaluation cell uses the box-label dataset (`data/df2_yolo`), so its numbers are directly comparable with `results/detection/det_test_metrics.json`.

# ShopTheLook · 01 · Train the garment detector (Colab)

Fine-tunes **YOLO11n** (COCO-pretrained) on a fixed 15,000-image subset of DeepFashion2. It then
evaluates on the held-out **det_test** split (6,005 images, 9,744 boxes).

**Before you run**
1. Upload `shopthelook_code.zip` to Google Drive at `MyDrive/shopthelook/shopthelook_code.zip`.
2. Choose *Runtime → Change runtime type → T4 GPU*.
3. Choose *Runtime → Run all*. Accept the Google Drive permission prompt.

**Disconnects are OK.** Training checkpoints are written to Drive (`MyDrive/shopthelook/runs/`).
If Colab disconnects, reconnect and *Run all* again. Data prep is rebuilt (~15–25 min), and
training **resumes** from the last saved epoch.

**What to send back:** `MyDrive/shopthelook/shopthelook_phase2_outputs.zip` (made by the last cell).

All randomness is seeded (seed 42). The image lists for every split come from `splits/`
inside the code zip, so this notebook trains on exactly the same images as the local repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive and unpack the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/shopthelook'
REPO = '/content/shopthelook'
import os
assert os.path.exists(f'{DRIVE}/shopthelook_code.zip'), 'Upload shopthelook_code.zip to MyDrive/shopthelook/ first'
!rm -rf {REPO} && mkdir -p {REPO} && unzip -q -o {DRIVE}/shopthelook_code.zip -d {REPO}
%cd {REPO}
!ls

## 2. Install dependencies
Colab already ships PyTorch with CUDA. We pin `ultralytics` to the version used locally.

In [ ]:
!pip install -q ultralytics==8.4.215
import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Data
* `validation.zip` (1.8 GB) is downloaded in full. It supplies det_val (checkpoint selection) and det_test (reported metrics).
* From the 10.6 GB `train.zip`, **only the 15,000 listed images** are fetched with HTTP range requests (~0.9 GB).

Source: unofficial unencrypted Hugging Face re-upload of DeepFashion2 (`sahirp/deepfashion2`). See `docs/DATA.md` for licensing notes.

In [ ]:
!mkdir -p data/raw/df2
!test -s data/raw/df2/validation.zip || curl -L --retry 5 -o data/raw/df2/validation.zip https://huggingface.co/datasets/sahirp/deepfashion2/resolve/main/validation.zip
!ls -lh data/raw/df2

In [ ]:
!python -m src.data.prepare_df2 --train-subset 15000 --workers 32 --val --yolo 2>&1 | grep -vE 'it/s]|s/it]'

In [ ]:
# Sanity check: these must be exactly 15000 / 1510 / 6005 (the same subsets as the local repo)
for s, expected in [('train', 15000), ('val', 1510), ('test', 6005)]:
    n = len(os.listdir(f'data/df2_yolo/labels/{s}'))
    print(f'{s:5s}: {n:,} label files', 'OK' if n == expected else f'MISMATCH (expected {expected:,})')
    assert n == expected

## 4. Train
YOLO11n at 640 px, batch 32, up to 30 epochs. Training early-stops after 10 epochs without improvement on det_val.
The best checkpoint is chosen on **det_val**. det_test is never seen during training.
Expected time on a T4 is roughly 1.5–3 hours, depending on how fast Colab's CPUs load data (an estimate, not measured).

In [ ]:
!python -m src.train_detector --project {DRIVE}/runs --name yolo11n_df2_15k --device 0 --workers 2 --batch 32 --epochs 30

## 5. Evaluate on held-out det_test

In [ ]:
RUN = f'{DRIVE}/runs/yolo11n_df2_15k'
OUT = f'{DRIVE}/outputs'
!python -m src.eval.detection_eval --weights {RUN}/weights/best.pt --device 0 --out {OUT}/results/detection --tag det_test

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/results/detection/det_test_metrics.json')), indent=2))
pd.read_csv(f'{OUT}/results/detection/det_test_per_class.csv')

## 6. Package outputs for the local repo

In [ ]:
!mkdir -p {OUT}/models {OUT}/results/detection/train_run
!cp {RUN}/weights/best.pt {OUT}/models/detector_best.pt
!cp {RUN}/results.csv {RUN}/args.yaml {OUT}/results/detection/train_run/
!cp {RUN}/*.png {RUN}/*.jpg {OUT}/results/detection/train_run/ 2>/dev/null; true
!cd {OUT} && rm -f ../shopthelook_phase2_outputs.zip && zip -qr ../shopthelook_phase2_outputs.zip models results
!ls -lh {DRIVE}/shopthelook_phase2_outputs.zip
print('Done. Download MyDrive/shopthelook/shopthelook_phase2_outputs.zip and place it in the Myntra_project1 folder.')